In [2]:
import ROOT
# print(f"ROOT version: {ROOT.__version__}")
import os 
import time 
import numpy as np
from scipy.special import j0
import plotly.graph_objects as go
from scipy.integrate import fixed_quad


In [34]:
save_folder = 'run8'
n_points = 10000

n=3

lower_factor = 0.99
upper_factor = 2 - lower_factor

b_max = 30
q_max = 0.2



In [35]:
def read_data_file(filename):
    """Read data file and return arrays for x, y, y_error"""
    x_vals = []
    y_vals = []
    y_errs = []
    
    with open(filename, 'r') as f:
        for line in f:
            if line.strip() and not line.startswith('#'):
                parts = line.split()
                if len(parts) >= 3:
                    x_vals.append(float(parts[0]))
                    y_vals.append(float(parts[1]))
                    y_errs.append(float(parts[2]))
    
    return x_vals, y_vals, y_errs

In [36]:
# Load experimental data - CHANGED to ROOT's TGraphErrors
x_atlas_all, y_atlas_all, yerr_atlas_all = read_data_file('../../../data/ens_atlas_difc0_2.dat')
x_totem_all, y_totem_all, yerr_totem_all = read_data_file('../../../data/ens_totem_difc0_2.dat')

# Function to process data for each experiment - CHANGED for ROOT objects
def process_data(x_data, y_data, yerr_data, energy_blocks):
    x_values = []
    y_values = []
    y_errors = []
    
    for start, end in energy_blocks:
        if end is None:
            end = len(x_data)
        x_values.append(x_data[start:end])
        y_values.append(y_data[start:end])
        y_errors.append(yerr_data[start:end])
    
    return x_values, y_values, y_errors


In [37]:
atlas_blocks = [(0, 29), (29, 58), (58, None)]
totem_blocks = [(0, 65), (65, 118), (118, None)]

# Process data - CHANGED to use the lists
x_atlas, y_atlas, yerr_atlas = process_data(x_atlas_all, y_atlas_all, yerr_atlas_all, atlas_blocks)
x_totem, y_totem, yerr_totem = process_data(x_totem_all, y_totem_all, yerr_totem_all, totem_blocks)

# Extract values by energy - SAME
x_7_atlas, y_7_atlas, yerr_7_atlas = x_atlas[0], y_atlas[0], yerr_atlas[0]
x_8_atlas, y_8_atlas, yerr_8_atlas = x_atlas[1], y_atlas[1], yerr_atlas[1]
x_13_atlas, y_13_atlas, yerr_13_atlas = x_atlas[2], y_atlas[2], yerr_atlas[2]

In [38]:
# print(x_7_atlas, y_7_atlas, yerr_7_atlas)
# print(x_8_atlas, y_8_atlas, yerr_8_atlas)
# print(x_13_atlas, y_13_atlas, yerr_13_atlas)

In [39]:
import math

b_0 = (33 - 6) / (12 * math.pi)
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0
s0 = 1.0
alpha_prime = 0.25

ensemble_parameters = {
    'atlas': {
        'log': {
            'eps': 0.0753,
            'mg': 0.356,
            'a1': 1.373,
            'a2': 2.50
        },
        'pl': {
            'eps': 0.0753,
            'mg': 0.421,
            'a1': 1.517,
            'a2': 2.05
        }
    }
}

# Get parameters for selected configuration
initial_params_log_atlas = ensemble_parameters['atlas']['log']
initial_params_pl_atlas = ensemble_parameters['atlas']['pl']

In [40]:
import math

def m2_log(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = math.log((q2 + rho_mg_squared) / lambda_squared) / math.log(rho_mg_squared / lambda_squared)
    return mg ** 2 * ratio ** (-1 - gamma_1)

def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = math.log((q2 + rho_mg_squared) / lambda_squared) / math.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)

def G_p(q2, a1, a2):
    return math.exp(-(a1 * q2 + a2 * q2 ** 2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * math.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, phi, mg, a1, a2, m2_func, q):
    q2 = q ** 2
    qk_cos = q * k * math.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1, a2)
    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, phi, mg, a1, a2, m2_func, q):
    q2 = q ** 2
    qk_cos = q * k * math.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)

def born_sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323

def born_amp(diff_T, s, epsilon, t):
    
    alpha_pomeron = 1.0 + epsilon + 0.25 * t

    first_term = s**(alpha_pomeron)
    second_term = 1/(1**(alpha_pomeron-1))
    
    regge_factor = first_term * second_term
    return 1j * 8.0 * regge_factor * diff_T

In [41]:
def integrate_adaptive_singular(func_expression, x_min, x_max, tolerance=1e-8):
    # Cria integrador com ADAPTIVESINGULAR
    integrator = ROOT.Math.IntegratorOneDim(ROOT.Math.IntegrationOneDim.kADAPTIVESINGULAR)
    
    # Define tolerâncias
    integrator.SetAbsTolerance(tolerance)
    integrator.SetRelTolerance(1e-6)  # Mantém a mesma usada no código original
    
    # Associa função
    integrator.SetFunction(func_expression)
    
    # Executa integração
    result = integrator.Integral(x_min, x_max)
    error = integrator.Error()
    
    return result

In [42]:
import math
import cmath
import ROOT
from ROOT import Math

# -------------------------------
# Integral over phi
# -------------------------------
def phi_integral(k, mg, a1, a2, m2_func, q):
    def integrand(phi):
        return k * (T_1(k, phi, mg, a1, a2, m2_func, q) -
                    T_2(k, phi, mg, a1, a2, m2_func, q))

    func = ROOT.Math.Functor(integrand, 1)
    return integrate_adaptive_singular(func, 0.0, 2*math.pi)

# -------------------------------
# Integral over k
# -------------------------------
def k_integral(k, mg, a1, a2, m2_func, q):
    return phi_integral(k, mg, a1, a2, m2_func, q)

# -------------------------------
# Double integral over phi and k
# -------------------------------
def compute_k_phi_integral(sqrt_s_val, mg, a1, a2, m2_func, q):
    def k_func(k):
        return k_integral(k, mg, a1, a2, m2_func, q)

    func = ROOT.Math.Functor(k_func, 1)
    return integrate_adaptive_singular(func, 0.0, sqrt_s_val)

# -------------------------------
# Chi integral over q
# -------------------------------
def chi_integral(sqrt_s_val, b, q_max, born_amp_value):
    s = sqrt_s_val**2

    def q_integrand(q):
        return (q * j0(b * q) * born_amp_value) / s

    func_real = ROOT.Math.Functor(lambda q: q_integrand(q).real, 1)
    func_imag = ROOT.Math.Functor(lambda q: q_integrand(q).imag, 1)

    real_part = integrate_adaptive_singular(func_real, 0.0, q_max)
    imag_part = integrate_adaptive_singular(func_imag, 0.0, q_max)

    return real_part + 1j * imag_part

# -------------------------------
# Eikonal amplitude integral over b
# -------------------------------
def eik_amp(sqrt_s_values, b_max, q_max, born_amp_value):
    if isinstance(sqrt_s_values, (int, float)):
        sqrt_s_values = [sqrt_s_values]

    amp_list = []
    for sqrt_s_val in sqrt_s_values:
        s = sqrt_s_val**2

        def b_integrand(b):
            chi_val = chi_integral(sqrt_s_val, b, q_max, born_amp_value)
            return b * (1 - cmath.exp(1j * chi_val))

        func_real = ROOT.Math.Functor(lambda b: b_integrand(b).real, 1)
        func_imag = ROOT.Math.Functor(lambda b: b_integrand(b).imag, 1)

        real_part = integrate_adaptive_singular(func_real, 0.0, b_max)
        imag_part = integrate_adaptive_singular(func_imag, 0.0, b_max)

        A_eik = 1j * s * (real_part + 1j * imag_part)
        amp_list.append(A_eik)

    return amp_list if len(amp_list) > 1 else amp_list[0]

# -------------------------------
# Differential cross-section
# -------------------------------
def differential_sigma(amp_value, s):
    amp_squared = amp_value.imag**2
    denominator = 16 * math.pi * s**2
    return amp_squared / denominator * 0.389379323


In [43]:
import numpy as np

# escolha do modelo de massa
mass_model = 'pl'
m2_func = m2_pl if mass_model == 'pl' else m2_log

def model_function(x_born, eps, mg, a1, a2, sqrt_s, model_type='log'):
    """
    x_born: array com os valores experimentais de q^2
    eps, mg, a1, a2: parâmetros do modelo
    sqrt_s: energia
    model_type: 'pl' ou 'log'
    """
    m2_func = m2_log if model_type == 'log' else m2_pl
    s = sqrt_s ** 2
    results = []

    # loop sobre os pontos experimentais (para cada t)
    for q_exp in x_born:  
        t = -q_exp

        # integral sobre q (usando ROOT adaptativo em vez de np.trapz)
        def q_integrand(q):
            return compute_k_phi_integral(
                sqrt_s_val=s,
                mg=mg,
                a1=a1,
                a2=a2,
                m2_func=m2_func,
                q=q
            )

        func_q = ROOT.Math.Functor(q_integrand, 1)
        diff_T = integrate_adaptive_singular(func_q, 0.0, 0.1)  # mesmo range de lst_q_integration

        # amplitudes
        born_amplitude = born_amp(diff_T, s, eps, t)
        eik_amplitude = eik_amp(s, b_max, q_max, born_amplitude)

        # seção de choque diferencial
        diff_sigma = differential_sigma(eik_amplitude, s)
        results.append(diff_sigma)

    return np.array(results)


In [ ]:
import ROOT
import numpy as np
import time

# lista de valores de q para a integral eikonal
lst_q_integration = np.linspace(0, 0.1, 1000)

def model_function(x_born, eps, mg, a1, a2, sqrt_s, model_type='log'):
    """
    x_born: array com os valores experimentais de q^2
    eps, mg, a1, a2: parâmetros do modelo
    sqrt_s: energia
    model_type: 'pl' ou 'log'
    """
    m2_func = m2_log if model_type == 'log' else m2_pl
    s = sqrt_s ** 2
    results = []

    # loop sobre os pontos experimentais (para cada t)
    for q_exp in x_born:  
        t = -q_exp

        # cálculo do integrando em q da integral eikonal
        integrand_values = []
        for q_int in lst_q_integration:  
            T = compute_k_phi_integral(
                sqrt_s_val=s,
                mg=mg,
                a1=a1,
                a2=a2,
                m2_func=m2_func,
                q=q_int
            )
            integrand_values.append(T)

        # integral sobre q
        diff_T = np.trapz(integrand_values, lst_q_integration)

        # amplitudes
        born_amplitude = born_amp(diff_T, s, eps, t)
        eik_amplitude = eik_amp(s, b_max, q_max, born_amplitude)

        # seção de choque diferencial
        diff_sigma = differential_sigma(eik_amplitude, s)
        results.append(diff_sigma)

    return np.array(results)

# Global variables to store data for the cost function
current_data = {}

def chi2_function(params):
    """
    Cost function for ROOT minimizer
    params: array with [eps, mg, a1, a2]
    """
    eps, mg, a1, a2 = params[0], params[1], params[2], params[3]
    
    total_chi2 = 0.0
    
    # Loop through all datasets
    for dataset_key, dataset in current_data.items():
        x_data = dataset['x']
        y_data = dataset['y']
        yerr_data = dataset['yerr'] 
        energy = dataset['energy']
        model_type = dataset['model_type']
        
        # Calculate model predictions
        y_model = model_function(x_data, eps, mg, a1, a2, energy, model_type)
        
        # Calculate chi2 for this dataset
        chi2 = np.sum(((y_data - y_model) / yerr_data) ** 2)
        total_chi2 += chi2
    
    return total_chi2

def setup_data_log_atlas():
    """Setup data for log model with ATLAS datasets"""
    global current_data
    current_data = {
        '7tev': {
            'x': x_7_atlas,
            'y': y_7_atlas, 
            'yerr': yerr_7_atlas,
            'energy': 7000,
            'model_type': 'log'
        },
        '8tev': {
            'x': x_8_atlas,
            'y': y_8_atlas,
            'yerr': yerr_8_atlas, 
            'energy': 8000,
            'model_type': 'log'
        },
        '13tev': {
            'x': x_13_atlas,
            'y': y_13_atlas,
            'yerr': yerr_13_atlas,
            'energy': 13000,
            'model_type': 'log'
        }
    }

def setup_data_pl_atlas():
    """Setup data for pl model with ATLAS datasets"""
    global current_data
    current_data = {
        '7tev': {
            'x': x_7_atlas,
            'y': y_7_atlas,
            'yerr': yerr_7_atlas,
            'energy': 7000,
            'model_type': 'pl'
        },
        '8tev': {
            'x': x_8_atlas,
            'y': y_8_atlas,
            'yerr': yerr_8_atlas,
            'energy': 8000,
            'model_type': 'pl'
        },
        '13tev': {
            'x': x_13_atlas,
            'y': y_13_atlas,
            'yerr': yerr_13_atlas,
            'energy': 13000,
            'model_type': 'pl'
        }
    }

def root_optimization(initial_params, model_type: str, ensemble: str,
                     minimizerName="Minuit2", algoName="Migrad"):
    """
    ROOT-based optimization function
    """
    print('\n')
    print(80 * '-')
    print(f"Iniciando otimização dos parâmetros usando ROOT para {model_type} em {ensemble.upper()}")

    start_time = time.time()
    
    # Setup data based on model type
    if model_type == 'log':
        setup_data_log_atlas()
    else:
        setup_data_pl_atlas()
    
    # Create minimizer
    minimizer = ROOT.Math.Factory.CreateMinimizer(minimizerName, algoName)
    if not minimizer:
        raise RuntimeError(f"Cannot create minimizer \"{minimizerName}\". Maybe the required library was not built?")

    # Set minimizer parameters based on model type
    if model_type == 'pl':
        minimizer.SetMaxFunctionCalls(1000)
        minimizer.SetMaxIterations(1000)
        minimizer.SetTolerance(1e-2)
        minimizer.SetPrintLevel(1)
        minimizer.SetStrategy(0)
    else:  # log
        minimizer.SetMaxFunctionCalls(70)
        minimizer.SetMaxIterations(70)
        minimizer.SetTolerance(1e-2)
        minimizer.SetPrintLevel(1)
        minimizer.SetStrategy(2)

    # Create function wrapper for minimizer
    f = ROOT.Math.Functor(chi2_function, 4)  # 4 parameters
    minimizer.SetFunction(f)

    # Set initial parameters and bounds
    param_names = ["eps", "mg", "a1", "a2"]
    initial_values = [initial_params['eps'], initial_params['mg'], 
                     initial_params['a1'], initial_params['a2']]
    step_sizes = [abs(val) * 0.01 for val in initial_values]  # 1% of initial value as step

    for i, (name, value, step) in enumerate(zip(param_names, initial_values, step_sizes)):
        minimizer.SetVariable(i, name, value, step)
        
        # Set bounds based on model type
        if model_type == 'pl':
            down = 0.80
            up = 2 - down
        else:  # log
            down = 0.94
            up = 2 - down
            
        lower_bound = down * value
        upper_bound = up * value
        
        # Special handling for specific parameters in log model
        if model_type == 'log' and name in ['eps', 'a1']:
            # Remove bounds for eps and a1 in log model (as in original code)
            pass
        else:
            minimizer.SetVariableLimits(i, lower_bound, upper_bound)

    # Perform minimization
    try:
        success = minimizer.Minimize()
        # Handle different return types from ROOT
        if isinstance(success, bool):
            minimization_success = success
        else:
            # Some ROOT versions return status codes
            minimization_success = (success == 0 or success == True)
    except Exception as e:
        print(f"Erro durante minimização: {e}")
        minimization_success = False

    if not minimization_success:
        print(f"Minimização pode ter falhado para {model_type} em {ensemble}")
        print(f"Valor final da função: {minimizer.MinValue()}")
        # Continue anyway to see results

    print(f'Finalizado a minimização para {model_type} em {ensemble}')

    execution_time = time.time() - start_time
    minutes = int(execution_time // 60)
    seconds = execution_time % 60
    print(f'Tempo de execução para {model_type} em {ensemble}: {minutes} min {seconds:.2f} s \n')

    print(f"Parâmetros otimizados para {model_type} em {ensemble}: \n")
    
    # Get results
    xs = minimizer.X()
    errors = minimizer.Errors() if minimizer.Errors() else [0]*4
    
    for i, name in enumerate(param_names):
        print(f'{name}: {xs[i]} ± {errors[i]}')
    
    # Calculate degrees of freedom
    ndof = sum(len(dataset['x']) for dataset in current_data.values()) - 4
    chi2_per_dof = minimizer.MinValue() / ndof
    print(f'chi2/ndof: {chi2_per_dof}')
    
    # Store results in a dictionary similar to iminuit format
    result = {
        'values': {name: xs[i] for i, name in enumerate(param_names)},
        'errors': {name: errors[i] for i, name in enumerate(param_names)},
        'fval': minimizer.MinValue(),
        'ndof': ndof,
        'success': minimization_success
    }

    return result

# Example usage:
if __name__ == "__main__":
    # Example initial parameters
    initial_params_example = {
        'eps': 1.0,
        'mg': 0.5,
        'a1': 2.0,
        'a2': 1.5
    }
    
    # Run optimization for log model
    # result_log = root_optimization(initial_params_example, 'log', 'atlas')
    
    # Run optimization for pl model  
    result_pl = root_optimization(initial_params_pl_atlas, 'pl', 'atlas')



--------------------------------------------------------------------------------
Iniciando otimização dos parâmetros usando ROOT para pl em ATLAS


/tmp/ipykernel_19970/1899616675.py:37: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  diff_T = np.trapz(integrand_values, lst_q_integration)
